# Actividad S2 M9 — 02/09/2025




In [2]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
from utilidades.tools import *
# from utilidades import tools_extra  # descomenta si usarás extras

# Recarga automática si editas módulos
%load_ext autoreload
%autoreload 2

print("Entorno listo (utilidades.tools): pandas, numpy, sklearn, statsmodels, etc.")


Entorno listo (utilidades.tools): pandas, numpy, sklearn, statsmodels, etc.


ACTIVIDAD SESIÓN APACHE SPARK
Imagina que trabajas para una empresa de análisis de mercado y tu tarea es estudiar las preferencias
de los consumidores en Sudamérica en cuanto a los modelos de teléfonos inteligentes. Para esto,
se te ha entregado un conjunto de datos que contiene información sobre las marcas y modelos de
teléfonos más vendidos en distintos países de Sudamérica, junto con la edad de los compradores y
las características del teléfono (como la cantidad de memoria RAM, la capacidad de la batería, el
precio, etc.).

INSTRUCCIONES
1.- Instalación y configuración de PySpark (1 punto)

- Configura correctamente el entorno de PySpark y crea una SparkSession con el nombre
AnalisisTelefonos.


In [3]:
# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("Analisis_Telefonos") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/03 13:53:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2.- Carga de datos (1 punto)

- Carga el archivo CSV proporcionado (con el nombre telefonos_sudamerica.csv) en un
DataFrame de PySpark. Asegúrate de que el archivo contenga los encabezados.


In [4]:
# Cargar con Spark
df = spark.read.option("header", True).option("inferSchema", True).csv("telefonos_sudamerica.csv")
print("Filas:", df.count())
df.printSchema()


Filas: 10
root
 |-- pais: string (nullable = true)
 |-- modelo: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- precio: integer (nullable = true)
 |-- memoria_ram: integer (nullable = true)
 |-- capacidad_bateria: integer (nullable = true)
 |-- edad_comprador: integer (nullable = true)
 |-- fecha_venta: date (nullable = true)



3.- Exploración inicial de los datos (1 punto)

- Muestra las primeras 10 filas del DataFrame y realiza una inspección básica de los tipos de
datos de cada columna. ¿Existen valores nulos o erróneos en alguna columna?

In [5]:
# Inspeccion de datos
df.show(10, truncate=False)
print("=== Nulos por columna ===")
for c in df.columns:
    print(f"{c:22s} ->", df.filter(F.col(c).isNull()).count())
#
print("No existen valores nulos ni erróneros en el dataframe")
df.printSchema()

+---------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|pais     |modelo       |marca   |precio|memoria_ram|capacidad_bateria|edad_comprador|fecha_venta|
+---------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|Argentina|Galaxy S21   |Samsung |1000  |8          |4000             |30            |2023-01-15 |
|Brasil   |iPhone 13    |Apple   |1200  |6          |3500             |25            |2023-02-10 |
|Chile    |Xiaomi Mi 11 |Xiaomi  |800   |6          |4500             |28            |2023-03-05 |
|Brasil   |Galaxy A72   |Samsung |700   |6          |5000             |22            |2023-02-12 |
|Peru     |Redmi Note 10|Xiaomi  |350   |4          |4000             |34            |2023-01-20 |
|Colombia |iPhone 12    |Apple   |950   |6          |3500             |27            |2023-02-18 |
|Brasil   |Motorola Edge|Motorola|750   |8          |5000             |31            |2023-01-25 |
|Argentina

4.- Filtrado de datos (2 puntos)

- Filtra el DataFrame para obtener únicamente los teléfonos vendidos en Brasil.

- Luego, filtra esos datos para obtener solo los teléfonos de la marca Samsung.


In [6]:
from pyspark.sql import functions as F
df_bra=df.filter(F.col("pais") == "Brasil")
print("Registros en telefonos vendidos en brasil:", df_bra.count())
df_bra.show()

Registros en telefonos vendidos en brasil: 3
+------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|  pais|       modelo|   marca|precio|memoria_ram|capacidad_bateria|edad_comprador|fecha_venta|
+------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|Brasil|    iPhone 13|   Apple|  1200|          6|             3500|            25| 2023-02-10|
|Brasil|   Galaxy A72| Samsung|   700|          6|             5000|            22| 2023-02-12|
|Brasil|Motorola Edge|Motorola|   750|          8|             5000|            31| 2023-01-25|
+------+-------------+--------+------+-----------+-----------------+--------------+-----------+



In [7]:
df_bra_sams=df_bra.filter(F.col("marca") == "Samsung")
print("Registros en telefonos vendidos en brasil de marca Samsung:", df_bra_sams.count())
df_bra_sams.show()

Registros en telefonos vendidos en brasil de marca Samsung: 1
+------+----------+-------+------+-----------+-----------------+--------------+-----------+
|  pais|    modelo|  marca|precio|memoria_ram|capacidad_bateria|edad_comprador|fecha_venta|
+------+----------+-------+------+-----------+-----------------+--------------+-----------+
|Brasil|Galaxy A72|Samsung|   700|          6|             5000|            22| 2023-02-12|
+------+----------+-------+------+-----------+-----------------+--------------+-----------+



5.- Operaciones de agrupamiento y agregación (2 puntos)

- Agrupa los datos por país y calcula la venta promedio de los teléfonos (promedio de precio).

- Agrupa los datos por marca y calcula el número de teléfonos vendidos por cada marca.


In [8]:
agrupado_pais=(df
    .groupBy("pais")
    .agg(
        F.avg("precio").alias("precio_promedio"))
    .orderBy("pais")
    )
agrupado_pais.show()

+---------+-----------------+
|     pais|  precio_promedio|
+---------+-----------------+
|Argentina|            750.0|
|   Brasil|883.3333333333334|
|    Chile|            600.0|
| Colombia|            950.0|
|     Peru|            725.0|
+---------+-----------------+



6.- Análisis por rango de edad (1 punto)

- Crea una nueva columna en el DataFrame que agrupe a los compradores por rango de edad
(por ejemplo, 18-25 años, 26-35 años, 36-50 años, 51+ años).

- Agrupa los datos por este nuevo rango de edad y muestra el promedio de precio de los
teléfonos vendidos para cada rango de edad.



In [9]:
df.withColumn(
    "rango_edad",
    F.when((F.col("edad_comprador") >= 18) & (F.col("edad_comprador") <= 25), "18-25 años")
     .when ((F.col("edad_comprador") >= 26) & (F.col("edad_comprador") <= 35), "26-35 años")
     .when ((F.col("edad_comprador") >=36) & (F.col("edad_comprador") <= 50), "36-50 años")
     .otherwise("51+ años")
).groupBy("rango_edad").agg(F.avg("precio").alias("precio_promedio")).show()

+----------+-----------------+
|rango_edad|  precio_promedio|
+----------+-----------------+
|26-35 años|764.2857142857143|
|18-25 años|            800.0|
+----------+-----------------+



7.- Análisis de correlación (1 punto)

- Calcula la correlación entre las columnas memoria_ram y precio. ¿Qué tipo de correlación
existe entre estas dos variables?


In [20]:
corr=df.stat.corr("precio","memoria_ram", method="pearson")
print(
    f"La correlación entre el precio y la memoria ram es de {corr:.2f} "
    "por ende es una alta correlación positiva"
)

La correlación entre el precio y la memoria ram es de 0.63 por ende es una alta correlación positiva


8.- Filtrado por características del teléfono (1 punto)

- Filtra los teléfonos que tengan una memoria RAM mayor a 6 GB y una batería mayor a 4000
mAh. ¿Cuántos teléfonos cumplen con esta condición?


In [26]:
df_ram_bat=df.filter((F.col("memoria_ram")>6)&(F.col("capacidad_bateria")>4000))

df_ram_bat.show()
print(f"Los telefonos que se vendieron con más de 6 GB de ram y con la batería mayor a 4000 mAH son: {df_ram_bat.count()}")

+------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|  pais|       modelo|   marca|precio|memoria_ram|capacidad_bateria|edad_comprador|fecha_venta|
+------+-------------+--------+------+-----------+-----------------+--------------+-----------+
|Brasil|Motorola Edge|Motorola|   750|          8|             5000|            31| 2023-01-25|
|  Peru|   Galaxy S21| Samsung|  1100|          8|             4500|            33| 2023-02-22|
+------+-------------+--------+------+-----------+-----------------+--------------+-----------+

Los telefonos que se vendieron con más de 6 GB de ram y con la batería mayor a 4000 mAH son: 2


9.- Guardar los resultados (1 punto)

- Guarda el DataFrame resultante de los filtros y agregaciones anteriores en un nuevo archivo
CSV llamado resultados_analisis.csv.


In [34]:
(df_bra
    .coalesce(1)   # une todas las particiones en 1
    .write
    .mode("overwrite")
    .option("header", True)
    .csv("resultados_analisis.csv")
)

In [43]:
import pyspark.sql.functions as F
import shutil
import os

# Partimos del DataFrame df_descarga
df_descarga = df_bra

# 1. Nombre del archivo 
archivo_salida="resultado_analisis"

# 2. Guardamos en carpeta (con 1 sola partición)
output_dir = f"{archivo_salida}_temp"


(
    df_descarga
    .coalesce(1)   # fuerza 1 archivo de salida
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(output_dir)
)

# 3. Renombrar a un único archivo CSV limpio
for file in os.listdir(output_dir):
    if file.startswith("part") and file.endswith(".csv"):
        shutil.move(os.path.join(output_dir, file), f"{archivo_salida}.csv")

# 4. Eliminar la carpeta temporal
shutil.rmtree(output_dir)

print(f"✅ Archivo guardado como {archivo_salida}.csv")

✅ Archivo guardado como resultado_analisis.csv


25/09/03 15:26:31 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 748589 ms exceeds timeout 120000 ms
25/09/03 15:26:31 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/03 15:26:38 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:669)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1296)
	at o